
## A. Reflex Agent for Abnormal Crowd Movement

A simple reflex agent uses the current percept only

Percepts:
- crowd movement/anomaly score
- crowd density
- emergency flag

Rules:
- If emergency flag is true → raise emergency alert
- Else if anomaly score exceeds threshold → activate high-detail monitoring and alert security
- Else → remain in low power monitoring mode

Eco-friendly behavior:
- Low power monitoring is used normally
- High resolution processing is activated only when abnormal movement is detected
- This reduces unnecessary compute, electricity use, and therefore indirect CO2 / water footprint.


In [ ]:
class ReflexCrowdAgent:
    def __init__(self, anomaly_threshold=0.75):
        self.anomaly_threshold = anomaly_threshold

    def act(self, anomaly_score, emergency=False):
        if emergency:
            return {
                "action": "EMERGENCY_ALERT",
                "power_mode": "HIGH",
                "co2_score": 3.0,
                "water_score": 0.3
            }

        if anomaly_score >= self.anomaly_threshold:
            return {
                "action": "ALERT_SECURITY_AND_HIGH_DETAIL_MONITORING",
                "power_mode": "HIGH",
                "co2_score": 2.0,
                "water_score": 0.2
            }

        return {
            "action": "LOW_POWER_MONITORING",
            "power_mode": "LOW",
            "co2_score": 0.5,
            "water_score": 0.05
        }

agent_A = ReflexCrowdAgent()

tests_A = [
    {"anomaly_score": 0.25, "emergency": False},
    {"anomaly_score": 0.82, "emergency": False},
    {"anomaly_score": 0.40, "emergency": True},
]

for x in tests_A:
    print(x, "->", agent_A.act(**x))

{'anomaly_score': 0.25, 'emergency': False} -> {'action': 'LOW_POWER_MONITORING', 'power_mode': 'LOW', 'co2_score': 0.5, 'water_score': 0.05}
{'anomaly_score': 0.82, 'emergency': False} -> {'action': 'ALERT_SECURITY_AND_HIGH_DETAIL_MONITORING', 'power_mode': 'HIGH', 'co2_score': 2.0, 'water_score': 0.2}
{'anomaly_score': 0.4, 'emergency': True} -> {'action': 'EMERGENCY_ALERT', 'power_mode': 'HIGH', 'co2_score': 3.0, 'water_score': 0.3}


The action is directly selected from a rule using the current percept:

Percept -> Rule -> Action

There is no planning or learning.



## B. Utility-Based Agent for Stressed / Tensed Patients

A utility-based agent evaluates several possible actions and selects the one with the highest utility

For a clinically acceptable action:


U(a) = w_r(Stress Relief) + w_s(Safety) - w_e(Energy) - w_c(CO_2) - w_w(Water)


Health and safety are given much larger weights than environmental cost

Possible actions:
- breathing guidance
- quiet-room support
- call medical staff
- emergency response

Eco friendly behavior:
- Low resource intervention is preferred when it is sufficient
- Stronger interventions are selected when patient condition requires them
- Necessary medical care is never blocked for environmental reasons


In [ ]:
class UtilityPatientAgent:
    def __init__(self):
        self.actions = {
            "BREATHING_GUIDANCE": {
                "relief": 5, "safety": 6, "energy": 1, "co2": 0.5, "water": 0.0
            },
            "QUIET_ROOM_SUPPORT": {
                "relief": 7, "safety": 7, "energy": 2, "co2": 1.0, "water": 0.1
            },
            "CALL_MEDICAL_STAFF": {
                "relief": 8, "safety": 9, "energy": 3, "co2": 1.5, "water": 0.1
            },
            "EMERGENCY_RESPONSE": {
                "relief": 10, "safety": 10, "energy": 5, "co2": 3.0, "water": 0.2
            }
        }

    def choose_action(self, stress_level, danger=False):
        # Hard safety constraint
        if danger or stress_level >= 9:
            return "EMERGENCY_RESPONSE", None

        scores = {}
        for action, a in self.actions.items():
            if action == "EMERGENCY_RESPONSE":
                continue

            # Higher stress makes strong relief more valuable.
            relief_weight = 1.0 + stress_level / 10.0

            utility = (
                relief_weight * a["relief"]
                + 1.5 * a["safety"]
                - 0.4 * a["energy"]
                - 0.5 * a["co2"]
                - 0.2 * a["water"]
            )
            scores[action] = utility

        best = max(scores, key=scores.get)
        return best, scores

agent_B = UtilityPatientAgent()

for stress in [3, 6, 8, 10]:
    action, scores = agent_B.choose_action(stress)
    print(f"Stress={stress}: {action}")
    if scores:
        print({k: round(v, 2) for k, v in scores.items()})


Stress=3: CALL_MEDICAL_STAFF
{'BREATHING_GUIDANCE': 14.85, 'QUIET_ROOM_SUPPORT': 18.28, 'CALL_MEDICAL_STAFF': 21.93}
Stress=6: CALL_MEDICAL_STAFF
{'BREATHING_GUIDANCE': 16.35, 'QUIET_ROOM_SUPPORT': 20.38, 'CALL_MEDICAL_STAFF': 24.33}
Stress=8: CALL_MEDICAL_STAFF
{'BREATHING_GUIDANCE': 17.35, 'QUIET_ROOM_SUPPORT': 21.78, 'CALL_MEDICAL_STAFF': 25.93}
Stress=10: EMERGENCY_RESPONSE


It does not follow only one fixed rule. It compares the expected desirability of multiple actions and chooses:


a = arg(max_a U(a))


The utility function balances patient benefit, safety, and environmental resource cost


## C. Goal-Based Agent for School Van

The main goal is: Reach school/home safely

Safety is a hard constraint. Only routes satisfying the security requirement are considered

Among safe routes, the agent minimizes a cost such as:

Cost = Distance + 2(Traffic) + 3(CO_2) + 1(Water)

The water term can represent the indirect water footprint of fuel/energy use and vehicle operation.

Eco-friendly behavior:
- reject unsafe routes first
- avoid excessive distance and idling
- among safe routes, prefer lower-emission and lower-resource routes


In [ ]:
class GoalSchoolVanAgent:
    def __init__(self, max_risk=3):
        self.max_risk = max_risk

    def choose_route(self, routes):
        # Goal constraint: only secure routes are feasible.
        feasible = [r for r in routes if r["risk"] <= self.max_risk]

        if not feasible:
            return None, "No route satisfies the safety goal."

        def route_cost(r):
            return (
                r["distance_km"]
                + 2.0 * r["traffic"]
                + 3.0 * r["co2_score"]
                + 1.0 * r["water_score"]
            )

        best = min(feasible, key=route_cost)
        return best, route_cost(best)

routes = [
    {
        "name": "Route A",
        "distance_km": 9,
        "risk": 2,
        "traffic": 4,
        "co2_score": 3,
        "water_score": 1.0
    },
    {
        "name": "Route B",
        "distance_km": 11,
        "risk": 1,
        "traffic": 2,
        "co2_score": 2,
        "water_score": 0.8
    },
    {
        "name": "Route C",
        "distance_km": 7,
        "risk": 5,    # Unsafe, rejected even though shorter
        "traffic": 1,
        "co2_score": 1,
        "water_score": 0.5
    }
]

agent_C = GoalSchoolVanAgent(max_risk=3)
best_route, cost = agent_C.choose_route(routes)

print("Selected route:", best_route)
print("Route cost:", cost)


Selected route: {'name': 'Route B', 'distance_km': 11, 'risk': 1, 'traffic': 2, 'co2_score': 2, 'water_score': 0.8}
Route cost: 21.8


The agent first checks whether an action sequence satisfies the goal:

Then it selects the best route among the goal-satisfying alternatives.

The unsafe Route C is rejected even though it is shorter. This shows that security dominates eco optimization



## D. Learning Agent to Encourage Sports in Young People

A learning agent improves its recommendations from feedback.

Possible activities:
- walking
- cycling
- football
- basketball
- indoor gaming

The reward combines:

Reward = Engagement + HealthBenefit + EcoBonus - FacilityResourceCost

Personal hydration is not penalized. Water reduction refers only to unnecessary facility/operational resource usage.

A simple contextual learning demonstration can use incremental action-value learning:

Q_new(a)=Q_old(a)+ alpha(R-Q_old(a))

In [ ]:
class LearningSportsAgent:
    def __init__(self, activities, alpha=0.3):
        self.activities = activities
        self.alpha = alpha
        self.q = {a: 0.0 for a in activities}
        self.count = {a: 0 for a in activities}

    def learn(self, activity, engagement, health_benefit, eco_bonus, facility_cost):
        reward = (
            engagement
            + health_benefit
            + eco_bonus
            - facility_cost
        )

        old = self.q[activity]
        self.q[activity] = old + self.alpha * (reward - old)
        self.count[activity] += 1

        return reward

    def recommend(self):
        return max(self.q, key=self.q.get)

activities = ["Walking", "Cycling", "Football", "Basketball", "Indoor Gaming"]
agent_D = LearningSportsAgent(activities)

feedback = [
    ("Walking",        6, 6, 2, 0.5),
    ("Cycling",        8, 8, 3, 0.5),
    ("Football",       9, 8, 2, 1.0),
    ("Basketball",     7, 7, 2, 1.0),
    ("Indoor Gaming",  5, 2, 0, 2.0),
    ("Cycling",        9, 8, 3, 0.5),
    ("Football",       8, 8, 2, 1.0),
]

for row in feedback:
    reward = agent_D.learn(*row)
    print(f"{row[0]:14s} reward={reward:4.1f}")

print("\nLearned Q-values:")
for activity, value in agent_D.q.items():
    print(f"{activity:14s}: {value:.3f}")

print("\nRecommended activity:", agent_D.recommend())


Walking        reward=13.5
Cycling        reward=18.5
Football       reward=18.0
Basketball     reward=15.0
Indoor Gaming  reward= 5.0
Cycling        reward=19.5
Football       reward=17.0

Learned Q-values:
Walking       : 4.050
Cycling       : 9.735
Football      : 8.880
Basketball    : 4.500
Indoor Gaming : 1.500

Recommended activity: Cycling


Its behavior changes after receiving feedback. The agent begins without knowing the best activity and updates its internal action values from experience.

This allows it to personalize recommendations rather than using a permanent hand-written rule
